# The EM algorithm
EM written out in NumPy (the same code as `images/em_core.py`), checked against *Mathematics for Machine Learning*
Section 11.3 and against scikit-learn; the lower bound; random starts; and the hard-assignment limit.

In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy import stats
from sklearn.mixture import GaussianMixture

np.set_printoptions(precision=3, suppress=True)
# one seeded random generator for the whole notebook
rng = np.random.default_rng(0)


def weighted_dens(X, pi, mu, cov):
    # pi_k N(x_n | mu_k, cov_k) for every observation n and component k
    return np.column_stack([p * stats.multivariate_normal(m, c).pdf(X) for p, m, c in zip(pi, mu, cov)])


def loglik(X, pi, mu, cov):
    return np.log(weighted_dens(X, pi, mu, cov).sum(axis=1)).sum()


def e_step(X, pi, mu, cov):
    # responsibilities: each row of weighted densities divided by its sum
    W = weighted_dens(X, pi, mu, cov)
    return W / W.sum(axis=1, keepdims=True)


def m_step(X, R):
    Nk = R.sum(axis=0)
    # weighted means first, then covariances around the NEW means, then weights
    mu = (R.T @ X) / Nk[:, None]
    cov = np.array([((R[:, k, None] * (X - mu[k])).T @ (X - mu[k])) / Nk[k] for k in range(R.shape[1])])
    return Nk / len(X), mu, cov

## 1. The seven-observation example

In [2]:
X7 = np.array([-3, -2.5, -1, 0, 2, 4, 5.0])[:, None]
pi, mu, cov = np.ones(3) / 3, np.array([[-4.0], [0.0], [8.0]]), np.array([[[1.0]], [[0.2]], [[3.0]]])
print("iteration 0: log-likelihood", round(loglik(X7, pi, mu, cov), 3))
for it in range(1, 6):
    R = e_step(X7, pi, mu, cov)
    pi, mu, cov = m_step(X7, R)
    print(f"iteration {it}: weights {pi}, means {mu.ravel()}, variances {cov.ravel()}, "
          f"log-likelihood {loglik(X7, pi, mu, cov):.3f}")

iteration 0: log-likelihood -28.326
iteration 1: weights [0.294 0.287 0.419], means [-2.701 -0.403  3.704], variances [0.144 0.438 1.527], log-likelihood -14.410
iteration 2: weights [0.285 0.285 0.43 ], means [-2.75  -0.501  3.655], variances [0.063 0.273 1.6  ], log-likelihood -13.977
iteration 3: weights [0.286 0.284 0.431], means [-2.75  -0.504  3.647], variances [0.062 0.251 1.622], log-likelihood -13.973
iteration 4: weights [0.286 0.283 0.431], means [-2.75  -0.504  3.645], variances [0.062 0.251 1.627], log-likelihood -13.973
iteration 5: weights [0.286 0.283 0.431], means [-2.75  -0.504  3.645], variances [0.062 0.251 1.629], log-likelihood -13.973


## 2. Two dimensions: Old Faithful
Eruption duration and waiting time (minutes), saved from seaborn's `geyser` dataset.

In [3]:
import pandas as pd
X = pd.read_csv("data/old_faithful.csv")[["duration", "waiting"]].to_numpy(float)
# a deliberately poor start: centres where there are no eruptions
pi, mu, cov = np.ones(2) / 2, np.array([[2.0, 90.0], [4.5, 50.0]]), np.array([np.diag([1.0, 100.0])] * 2)
lls = [loglik(X, pi, mu, cov)]
for it in range(60):
    pi, mu, cov = m_step(X, e_step(X, pi, mu, cov))
    lls.append(loglik(X, pi, mu, cov))
print("log-likelihood at iterations 0, 1, 5, 10, 60:", np.round([lls[i] for i in (0, 1, 5, 10, 60)], 1))
print("weights", pi, "\nmeans\n", mu)
gm = GaussianMixture(2, n_init=5, random_state=0).fit(X)
print("scikit-learn log-likelihood:", round(gm.score(X) * len(X), 1))
fig = go.Figure(go.Scatter(y=lls, mode="lines+markers"))
fig.update_layout(template="simple_white", xaxis_title="iteration", yaxis_title="log-likelihood", height=350)
fig.show()

log-likelihood at iterations 0, 1, 5, 10, 60: [-2111.4 -1270.2 -1182.  -1130.3 -1130.3]
weights [0.644 0.356] 
means
 [[ 4.29  79.968]
 [ 2.036 54.479]]
scikit-learn log-likelihood: -1130.3


## 3. The log-likelihood never decreases, and the lower bound touches it

In [4]:
steps = np.diff(lls)
print("any change below -1e-9 in 60 iterations?", bool((steps < -1e-9).any()))
# B(theta; q) with q = responsibilities of the same theta, seven observations at the start
pi, mu, cov = np.ones(3) / 3, np.array([[-4.0], [0.0], [8.0]]), np.array([[[1.0]], [[0.2]], [[3.0]]])
W = weighted_dens(X7, pi, mu, cov)
q = W / W.sum(axis=1, keepdims=True)
# 0 log 0 counts as 0
B = np.sum(np.where(q > 0, q * (np.log(W) - np.log(np.where(q > 0, q, 1))), 0))
print("bound B =", round(B, 3), "  log-likelihood =", round(loglik(X7, pi, mu, cov), 3))
# with other weights q the bound is strictly lower
q_flat = np.ones_like(q) / 3
print("bound with q = 1/3 everywhere:", round(np.sum(q_flat * (np.log(W) - np.log(q_flat))), 3))

any change below -1e-9 in 60 iterations? False
bound B = -28.326   log-likelihood = -28.326
bound with q = 1/3 everywhere: -116.045


## 4. Local maxima: 20 random starts
First Old Faithful (every start ends at the same answer), then the Iris flowers (four features, three components),
where many starts get stuck. Each run starts its means at three randomly chosen observations
(scikit-learn's `init_params="random_from_data"`, one start per run).


In [5]:
def run_from(X, K, idx, iters=300):
    p, m, c = np.ones(K) / K, X[idx], np.array([np.cov(X.T)] * K)
    for it in range(iters):
        p, m, c = m_step(X, e_step(X, p, m, c))
    return round(loglik(X, p, m, c), 1)


print("Old Faithful:", sorted({run_from(X, 2, rng.choice(len(X), 2, replace=False)) for s in range(20)}))
from collections import Counter
from sklearn.datasets import load_iris
from sklearn.metrics import adjusted_rand_score
Xi, yi = load_iris(return_X_y=True)
finals = []
for s in range(20):
    g = GaussianMixture(3, covariance_type="full", init_params="random_from_data",
                        random_state=s, max_iter=1000, tol=1e-6).fit(Xi)
    finals.append((round(g.score(Xi) * len(Xi), 1), round(adjusted_rand_score(yi, g.predict(Xi)), 2)))
for (ll, a), n in sorted(Counter(finals).items(), reverse=True):
    print(f"log-likelihood {ll:7.1f}  ARI {a:.2f}  starts {n}")


Old Faithful: [np.float64(-1130.3)]


log-likelihood  -180.2  ARI 0.90  starts 9
log-likelihood  -186.6  ARI 0.72  starts 1
log-likelihood  -189.8  ARI 0.56  starts 2
log-likelihood  -192.8  ARI 0.53  starts 1
log-likelihood  -193.1  ARI 0.52  starts 1
log-likelihood  -194.1  ARI 0.51  starts 1
log-likelihood  -195.7  ARI 0.56  starts 1
log-likelihood  -196.5  ARI 0.56  starts 1
log-likelihood  -198.6  ARI 0.47  starts 1
log-likelihood  -203.5  ARI 0.49  starts 2


## 5. Hard assignment: responsibilities as sigma shrinks
Centres 0 and 4, equal weights, observation at x = 1.

In [6]:
for s in (2, 1, 0.5, 0.1):
    a, b = stats.norm(0, s).logpdf(1), stats.norm(4, s).logpdf(1)
    print(f"sigma = {s}: responsibility of the centre at 0 = {1 / (1 + np.exp(b - a)):.6f}")

sigma = 2: responsibility of the centre at 0 = 0.731059
sigma = 1: responsibility of the centre at 0 = 0.982014
sigma = 0.5: responsibility of the centre at 0 = 1.000000
sigma = 0.1: responsibility of the centre at 0 = 1.000000


## 6. The six-point example: guess, score, update, repeat
Points 1, 2, 3, 6, 7, 8; curves A and B start at 2 and 4, variance 4, weights 0.5. Seven rounds to convergence.

In [7]:
X6 = np.array([1, 2, 3, 6, 7, 8.0])
pi6, mu6, var6 = np.array([0.5, 0.5]), np.array([2.0, 4.0]), np.array([4.0, 4.0])


def weighted6(pi, mu, var):
    return pi * stats.norm(mu, np.sqrt(var)).pdf(X6[:, None])


def loglik6(pi, mu, var):
    return np.log(weighted6(pi, mu, var).sum(axis=1)).sum()
p = (pi6, mu6, var6)
for rnd in range(9):
    print(f"round {rnd}: centres {p[1].round(2)}, variances {p[2].round(2)}, weights {p[0].round(2)}, log-likelihood {loglik6(*p):.2f}")
    W = weighted6(*p)
    R = W / W.sum(axis=1, keepdims=True)
    N = R.sum(axis=0)
    m = (R * X6[:, None]).sum(axis=0) / N
    p = (N / 6, m, (R * (X6[:, None] - m) ** 2).sum(axis=0) / N)
# shares after round 1 (used in the Note: point 3 gives A 0.56; points 6, 7, 8 give B 0.85, 0.93, 0.97)
p1 = (np.array([0.372, 0.628]), np.array([2.695, 5.569]), np.array([3.937, 5.609]))
W = weighted6(*p1)
print((W / W.sum(axis=1, keepdims=True)).round(3).T)


round 0: centres [2. 4.], variances [4. 4.], weights [0.5 0.5], log-likelihood -15.82
round 1: centres [2.69 5.57], variances [3.94 5.61], weights [0.37 0.63], log-likelihood -14.15
round 2: centres [2.43 5.74], variances [2.83 5.27], weights [0.37 0.63], log-likelihood -13.93
round 3: centres [2.11 5.98], variances [1.51 4.53], weights [0.38 0.62], log-likelihood -13.35
round 4: centres [1.92 6.31], variances [0.69 3.34], weights [0.41 0.59], log-likelihood -12.68
round 5: centres [1.92 6.63], variances [0.64 2.06], weights [0.45 0.55], log-likelihood -12.20
round 6: centres [1.98 6.91], variances [0.66 1.01], weights [0.49 0.51], log-likelihood -11.58
round 7: centres [2. 7.], variances [0.67 0.67], weights [0.5 0.5], log-likelihood -11.46
round 8: centres [2. 7.], variances [0.67 0.67], weights [0.5 0.5], log-likelihood -11.46
[[0.759 0.674 0.557 0.152 0.075 0.032]
 [0.241 0.326 0.443 0.848 0.925 0.968]]
